In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

torch.manual_seed(0)
np.random.seed(0)

In [ ]:
class ToyModel(nn.Module):
    def __init__(self, n_features: int, n_hidden: int):
        super().__init__()
        # W has shape (n_hidden, n_features). Each column is the 'direction' the model uses
        # to represent that feature in hidden space.
        self.W = nn.Parameter(torch.empty(n_hidden, n_features))
        nn.init.xavier_normal_(self.W)
        self.b = nn.Parameter(torch.zeros(n_features))

    def forward(self, x):
        # x: (batch, n_features)
        hidden = x @ self.W.T          # (batch, n_hidden) -- encode
        out = hidden @ self.W + self.b # (batch, n_features) -- decode (tied weights)
        return F.relu(out)

In [ ]:
def generate_batch(batch_size: int, n_features: int, sparsity: float, device):
    """Returns a (batch_size, n_features) tensor of sparse non-negative features."""
    values = torch.rand(batch_size, n_features, device=device)
    mask   = torch.rand(batch_size, n_features, device=device) > sparsity
    return values * mask

# sanity check: at S=0.9, roughly 10% of entries should be nonzero
sample = generate_batch(1000, n_features=5, sparsity=0.9, device=device)
print(f'Fraction of nonzero entries: {(sample > 0).float().mean().item():.3f} (expected ~0.10)')